### Experiment 1: Word - context representation

In [1]:
corpus = [
    "the cat eats fish",
    "the dog eats fish",
    "the cat likes milk",
    "the dog likes meat"
]

corpus

['the cat eats fish',
 'the dog eats fish',
 'the cat likes milk',
 'the dog likes meat']

In [2]:
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer
vectorizer = CountVectorizer(token_pattern = r"(?u)\b\w+\b")
vectorizer.fit(corpus)

vocab = vectorizer.get_feature_names_out()
word2id = vectorizer.vocabulary_

print(f"Vocabulary size: {len(vocab)}")
print(f"Vocabulary: {vocab}")
print(word2id)

Vocabulary size: 8
Vocabulary: ['cat' 'dog' 'eats' 'fish' 'likes' 'meat' 'milk' 'the']
{'the': 7, 'cat': 0, 'eats': 2, 'fish': 3, 'dog': 1, 'likes': 4, 'milk': 6, 'meat': 5}


In [3]:
# Word-Context Matrix

def build_cooccurrence_matrix(corpus, vocab, word2id, window_size):
    V = len(vocab)
    X = np.zeros((V, V))
    
    for doc in corpus:
        tokens = doc.split()
        for i, target in enumerate(tokens):
            # Từ không có trong từ điển
            if target not in word2id: 
                continue
            target_id = word2id[target]
            
            # Context [i - window_size, i + window_size]
            start = max(0, i - window_size)
            end = min(len(tokens), i + window_size + 1)
            
            for j in range(start, end):
                if i != j:
                    context = tokens[j]
                    if context in word2id:
                        context_id = word2id[context]
                        X[target_id, context_id] += 1
    return X

In [5]:
windows = [1, 2, 5]
matrices = {}

for w in windows:
    X = build_cooccurrence_matrix(corpus, vocab, word2id, window_size=w)
    matrices[w] = X  
    
    matrix_size = X.shape
    non_zero_count = np.count_nonzero(X)
    
    print(f"Window = {w}:")
    print(f"Matrix size: {matrix_size}")
    print(f"nnz: {non_zero_count}\n")

Window = 1:
Matrix size: (8, 8)
nnz: 18

Window = 2:
Matrix size: (8, 8)
nnz: 30

Window = 5:
Matrix size: (8, 8)
nnz: 36



In [6]:
from sklearn.metrics.pairwise import cosine_similarity
word1, word2 = "cat", "dog"
id1, id2 = word2id[word1], word2id[word2]
for w in windows:
    X = matrices[w]
    
    # 1D => 2D
    v1 = X[id1].reshape(1, -1)
    v2 = X[id2].reshape(1, -1)
    
    # similarity
    sim = cosine_similarity(v1, v2)[0][0]
    
    print(f"Window = {w}: Similarity = {sim:.4f}")

Window = 1: Similarity = 1.0000
Window = 2: Similarity = 0.8750
Window = 5: Similarity = 0.8750
